# ✈️ Retards et annulations des vols aux États-Unis en 2015
## Analyse exploratoire des données (EDA)

**Contexte :** le directeur marketing veut comprendre rapidement ce qui provoque les retards et les annulations de vols.
**Question centrale :** quels sont les principaux facteurs qui influencent les retards et les annulations, et comment varient-ils d'une compagnie à l'autre et d'un aéroport à l'autre ?

**Sommaire**
1. Import des librairies
2. Chargement des données
3. Premier aperçu
4. Qualité des données : valeurs manquantes, doublons, valeurs aberrantes
5. Nettoyage et préparation
6. Analyse univariée
7. Analyse bivariée : retards et annulations selon le temps, les compagnies et les aéroports
8. Causes des retards
9. Corrélations
10. Conclusion et messages clés

## 1. Import des librairies
On utilise **pandas** pour manipuler les tableaux, **numpy** pour les calculs, et **matplotlib / seaborn / plotly** pour les graphiques.

In [4]:
import pandas as pd
import numpy as np


## 2. Chargement des données
Nous disposons de 3 fichiers :
| Fichier | Contenu |
|---|---|
| `flights.csv` | un vol par ligne (date, compagnie, aéroports, horaires, retards, annulation) |
| `airlines.csv` | code IATA → nom de la compagnie |
| `airports.csv` | code IATA → nom, ville, État, coordonnées GPS de l'aéroport |

⚠️ `flights.csv` compte près de 5,8 millions de lignes. On lit les horaires comme du texte pour garder le format HHMM.

In [12]:

time_cols = ["SCHEDULED_DEPARTURE", "DEPARTURE_TIME", "WHEELS_OFF", "WHEELS_ON","SCHEDULED_ARRIVAL", "ARRIVAL_TIME"]

df_flights = pd.read_csv(
    "data/flights.csv",
    dtype={col: str for col in time_cols} | {"ORIGIN_AIRPORT": str, "DESTINATION_AIRPORT": str},
    low_memory=False
)

df_airlines = pd.read_csv("data/airlines.csv")

df_airports = pd.read_csv("data/airports.csv")

print("flights:", df_flights.shape)
print("arlines :", df_airlines.shape)
print("airports :", df_airports.shape)




flights: (5819079, 31)
arlines : (14, 2)
airports : (322, 7)


## 3. Premier aperçu des données
Objectif : comprendre la structure de chaque table, c'est-à-dire sa taille, ses colonnes et le type de chaque colonne.

💻 .shape, .head(), .info(), .dtypes, .describe(), .nunique()



## 3.1 Fichier Flights.csv
( analyse de la strucutre de ce 1er fichier csv )

In [13]:
#FICHIER FLIGHTS.CSV

df_flights.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,0005,...,0408,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,0010,...,0741,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,0020,...,0811,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,0020,...,0756,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,0025,...,0259,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df_flights.info()

<class 'pandas.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 31 columns):
 #   Column               Dtype  
---  ------               -----  
 0   YEAR                 int64  
 1   MONTH                int64  
 2   DAY                  int64  
 3   DAY_OF_WEEK          int64  
 4   AIRLINE              str    
 5   FLIGHT_NUMBER        int64  
 6   TAIL_NUMBER          str    
 7   ORIGIN_AIRPORT       str    
 8   DESTINATION_AIRPORT  str    
 9   SCHEDULED_DEPARTURE  str    
 10  DEPARTURE_TIME       str    
 11  DEPARTURE_DELAY      float64
 12  TAXI_OUT             float64
 13  WHEELS_OFF           str    
 14  SCHEDULED_TIME       float64
 15  ELAPSED_TIME         float64
 16  AIR_TIME             float64
 17  DISTANCE             int64  
 18  WHEELS_ON            str    
 19  TAXI_IN              float64
 20  SCHEDULED_ARRIVAL    str    
 21  ARRIVAL_TIME         str    
 22  ARRIVAL_DELAY        float64
 23  DIVERTED             int64  
 24  CANCELLED

In [16]:
df_flights.describe().T

,count,mean,std,min,25%,50%,75%,max
YEAR,5819079.0,2015.000000,0.000000,2015.0,2015.0,2015.0,2015.0,2015.0
MONTH,5819079.0,6.524085,3.405137,1.0,4.0,7.0,9.0,12.0
DAY,5819079.0,15.704594,8.783425,1.0,8.0,16.0,23.0,31.0
DAY_OF_WEEK,5819079.0,3.926941,1.988845,1.0,2.0,4.0,6.0,7.0
FLIGHT_NUMBER,5819079.0,2173.092742,1757.063999,1.0,730.0,1690.0,3230.0,9855.0
DEPARTURE_DELAY,5732926.0,9.370158,37.080942,-82.0,-5.0,-2.0,7.0,1988.0
TAXI_OUT,5730032.0,16.071662,8.895574,1.0,11.0,14.0,19.0,225.0
SCHEDULED_TIME,5819073.0,141.685892,75.210582,18.0,85.0,123.0,173.0,718.0
ELAPSED_TIME,5714008.0,137.006189,74.211072,14.0,82.0,118.0,168.0,766.0
AIR_TIME,5714008.0,113.511628,72.230822,7.0,60.0,94.0,144.0,690.0


In [18]:
df_flights.nunique().sort_values()

YEAR                      1
CANCELLED                 2
DIVERTED                  2
CANCELLATION_REASON       4
DAY_OF_WEEK               7
MONTH                    12
AIRLINE                  14
DAY                      31
SECURITY_DELAY          154
TAXI_OUT                184
TAXI_IN                 185
SCHEDULED_TIME          550
AIR_SYSTEM_DELAY        570
ORIGIN_AIRPORT          628
DESTINATION_AIRPORT     629
WEATHER_DELAY           632
AIR_TIME                675
LATE_AIRCRAFT_DELAY     695
ELAPSED_TIME            712
AIRLINE_DELAY          1067
DEPARTURE_DELAY        1217
ARRIVAL_DELAY          1240
SCHEDULED_DEPARTURE    1321
DISTANCE               1363
SCHEDULED_ARRIVAL      1435
WHEELS_ON              1440
WHEELS_OFF             1440
ARRIVAL_TIME           1440
DEPARTURE_TIME         1440
TAIL_NUMBER            4897
FLIGHT_NUMBER          6952
dtype: int64

In [19]:
df_airlines.head(14)

,IATA_CODE,AIRLINE
0,UA,United Air Lines Inc.
1,AA,American Airlines Inc.
2,US,US Airways Inc.
3,F9,Frontier Airlines Inc.
4,B6,JetBlue Airways
5,OO,Skywest Airlines Inc.
6,AS,Alaska Airlines Inc.
7,NK,Spirit Air Lines
8,WN,Southwest Airlines Co.
9,DL,Delta Air Lines Inc.


In [20]:
df_airports.head()

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
3,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
4,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447


# Remarque :

# Constat :

# Action : 

#

### Ce que l'on observe
- *(à compléter : nombre de lignes et de colonnes, types qui posent problème…)*

## 4. Qualité des données
### 4.1 Valeurs manquantes
On compte les valeurs manquantes par colonne, en nombre et en pourcentage.
**À interpréter :** certaines absences sont logiques. Un vol annulé n'a pas d'heure de départ, et les colonnes de causes de retard (`WEATHER_DELAY`…) ne sont remplies que si le retard dépasse 15 minutes.

💻 df.isna().sum(), df.isna().mean()*100, et éventuellement une heatmap des valeurs manquantes



### 4.2 Doublons
On vérifie s'il existe des lignes en double.